In [1]:
from bs4 import BeautifulSoup as bs
import json

with open(r"builds.html", "r", encoding="utf-8") as f:
    soup = bs(f, "html.parser")

cols=soup.find_all("div","col-sm-6")
print(len(cols))

473


First, you should decide what to add into the app, i.e base-game and/or DLCs. Initially, I was thinking of exporting the whole html structure, but the structure is not compatible with our design in general. Instead, we'll do dictionaries again, and export. This time, I don't plan to use Agents to format the data, but instead I'll try to manually import the necessary info when a certain build is called.

Let's list what we want to extract:

- Youtube ID: 'https://www.youtube.com/watch?v=' `ID GOES HERE` 
- Title: all titles are in `label_name` under `class_name`.
- The list and the text: get_text from all `p` and `ul`-tags  

Once we have all, we can create a big dictionary, and let the app access when a build is chosen.


Extracting the youtube ID is straightforward. We only need the id of the correct class in the html:

In [ ]:
cols[204].find("div",class_="class_name").get("id")

Next, we extract the name of the builds. Yet, I want to format the titles a bit, so we remove unwanted bits( given in `strrem`) with a loop:

In [ ]:
bld_nm=[]
strrem=[]

for i in range(n1,n2):
    namecln=cols[i].find_all("label_name")[0].get_text(strip=True)
    for u in strrem:
        namecln=namecln.replace(u,"")
    bld_nm.append(namecln)

Lastly, we need to extract items to collect and the build guide paragraph. Scrapping these will follow a similar fashion as well. A loop over the p-tags will accumulate the text and another one over the li-tags collects the item list. Then, we just repeat this for all entries:

In [ ]:
bigbuild=[]

for i in range(n1,n2):
    # The name:
    strrem=[" "]
    namecln=cols[i].find_all("h3")[0].get_text(strip=True)
    for u in strrem:
        namecln=namecln.replace(u,"")
    title=namecln.capitalize()
    # Youtube ID:
    youtubeid=cols[i].find("div",class_="youtube").get("id")
    # Items:
    bld_items=[]
    for li in cols[i].find_all("li"):
        bld_items.append(li.get_text())
    # Build description:
    bld_desc=""
    for p in cols[i].find_all("p"):
        bld_desc +=p.get_text().replace("\xa0"," ")
    #Collect:
    build={
        "Name": title,
        "YoutubeID": youtubeid,
        "Items":bld_items,
        "Description":bld_desc
    }
    bigbuild.append(build)
with open ("bigbuild.json","w") as f:
    json.dump(bigbuild,f)